In [10]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Fetch Data (Try MySQL first, fallback to saved CSV if server drops connection)
try:
    host = "18.136.157.135"
    port = 3306
    database = "project_profit_analysis"
    table_name = "startup"
    username = "dm_team5"
    password = quote_plus("DM!$!Team!520@4!23&")

    connection_string = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
    engine = create_engine(connection_string, connect_args={"connect_timeout": 10})
    df = pd.read_sql(f"SELECT * FROM {table_name};", engine)
    df.to_csv("startup_profit_data.csv", index=False)
    print("Data fetched directly from MySQL!")
except Exception as e:
    print("MySQL server busy/closed, loading from local backup 'startup_profit_data.csv'...")
    df = pd.read_csv("startup_profit_data.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# 2. Data Preprocessing (One-Hot Encoding for 'State')
df_encoded = pd.get_dummies(df, columns=["State"], drop_first=True)

X = df_encoded.drop(columns=["Profit"])
y = df_encoded["Profit"]

# 3. Train-Test Split & Model Training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

# 4. Model Evaluation
y_pred = model.predict(X_test)
print("\n--- Model Performance ---")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

print("\nIntercept:", model.intercept_)
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})
print(coef_df)

# 5. Predict Profit for Given Input Features (Fixed Column Names)
input_data = pd.DataFrame({
    "RD_Spend": [21892.92, 23940.93],
    "Administration": [81910.77, 96489.63],
    "Marketing_Spend": [164270.70, 137001.10]
})

# Align columns with training features X
for col in X.columns:
    if col not in input_data.columns:
        input_data[col] = 0

input_data = input_data[X.columns]

predicted_profits = model.predict(input_data)
input_data["Predicted_Profit"] = predicted_profits

print("\n--- Predicted Profit for Given Inputs ---")
print(input_data[["RD_Spend", "Administration", "Marketing_Spend", "Predicted_Profit"]])

Data fetched directly from MySQL!
Dataset Shape: (50, 5)
    RD_Spend  Administration  Marketing_Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94

--- Model Performance ---
R2 Score: 0.8987
MAE: 6961.48
RMSE: 9055.96

Intercept: 54028.03959364582
           Feature  Coefficient
0         RD_Spend     0.805630
1   Administration    -0.068788
2  Marketing_Spend     0.029855
3    State_Florida   938.793006
4   State_New York     6.987760

--- Predicted Profit for Given Inputs ---
   RD_Spend  Administration  Marketing_Spend  Predicted_Profit
0  21892.92        81910.77         164270.7      70935.540232
1  23940.93        96489.63         137001.1      70768.483767
